# MedDiag — Entrenamiento Mejorado de Parkinson con SMOTE

**Autor:** Diana Carolina Huertas González  
**Proyecto:** MedDiag - Sistema de Diagnóstico Médico Basado en IA  
**Dataset:** UCI ML Parkinson's Dataset  
**Objetivo:** Mejorar el modelo XGBoost usando SMOTE para balancear clases y reducir falsos positivos.

---
**Versión local** — Adaptada para ejecutar en Jupyter local (sin Google Colab)

---
## Celda 1 — Verificar dependencias

In [ ]:
# Celda 1 — Verificar que las dependencias están instaladas
try:
    import imblearn
    import xgboost
    import sklearn
    import matplotlib
    import seaborn
    import pandas
    import numpy
    print("✓ Todas las dependencias están instaladas")
    print(f"  imbalanced-learn: {imblearn.__version__}")
    print(f"  xgboost:         {xgboost.__version__}")
    print(f"  scikit-learn:    {sklearn.__version__}")
    print(f"  pandas:          {pandas.__version__}")
    print(f"  numpy:           {numpy.__version__}")
except ImportError as e:
    print(f"✗ Falta dependencia: {e}")
    print("Ejecuta: pip install imbalanced-learn xgboost scikit-learn matplotlib seaborn pandas")

---
## Celda 2 — Cargar el dataset UCI

In [ ]:
# Celda 2
import pandas as pd
import numpy as np
import ssl

# El certificado SSL de UCI ha expirado, usamos HTTP en su lugar
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"

# Intentar con HTTPS primero, si falla usar HTTP
try:
    df = pd.read_csv(url)
except Exception:
    print("HTTPS falló, intentando con HTTP...")
    url_http = "http://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"
    df = pd.read_csv(url_http)

print(f"Shape: {df.shape}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nDistribución de clases:")
print(df['status'].value_counts())
print(f"\nProporción: {df['status'].value_counts(normalize=True).round(3)}")

# Separar features y target
X = df.drop(columns=['name', 'status'])
y = df['status']

print(f"\nFeatures shape: {X.shape}")
print(f"Clases: 0=sano, 1=Parkinson")

---
## Celda 3 — Visualizar el desbalance (antes y después de SMOTE)

In [ ]:
# Celda 3
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Ver distribución original
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribución de clases: antes y después de SMOTE', fontsize=14)

# Antes
counts_orig = y.value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Sano (0)', 'Parkinson (1)'], counts_orig[[0,1]], color=colors, alpha=0.8, edgecolor='white')
axes[0].set_title(f'Original: {counts_orig[0]} sanos / {counts_orig[1]} PD')
axes[0].set_ylabel('Muestras')
for i, v in enumerate(counts_orig[[0,1]]):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Aplicar SMOTE para mostrar después
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(sampling_strategy=0.8, k_neighbors=5, random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y)

counts_smote = pd.Series(y_res).value_counts()
axes[1].bar(['Sano (0)', 'Parkinson (1)'], counts_smote[[0,1]], color=colors, alpha=0.8, edgecolor='white')
axes[1].set_title(f'Con SMOTE: {counts_smote[0]} sanos / {counts_smote[1]} PD')
for i, v in enumerate(counts_smote[[0,1]]):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nMuestras originales: {len(y)}")
print(f"Muestras después de SMOTE: {len(y_res)}")
print(f"Muestras sintéticas generadas: {len(y_res) - len(y)}")

---
## Celda 4 — Entrenar con SMOTE + validación cruzada

In [ ]:
# Celda 4
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, recall_score, roc_auc_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Pipeline completo: escalado → SMOTE → XGBoost
pipeline_smote = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.8,  # sanos = 80% del total de PD
        k_neighbors=5,
        random_state=42
    )),
    ('model', XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

# Validación cruzada estratificada (5-fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'f1':      make_scorer(f1_score),
    'recall':  make_scorer(recall_score),
    'roc_auc': make_scorer(roc_auc_score, needs_proba=True)
}

print("Ejecutando validación cruzada 5-fold...")
results = cross_validate(pipeline_smote, X, y, cv=cv, scoring=scoring, return_train_score=False)

print("\n=== Resultados con SMOTE (5-fold CV) ===")
print(f"F1-score : {results['test_f1'].mean():.3f} ± {results['test_f1'].std():.3f}")
print(f"Recall   : {results['test_recall'].mean():.3f} ± {results['test_recall'].std():.3f}")
print(f"AUC-ROC  : {results['test_roc_auc'].mean():.3f} ± {results['test_roc_auc'].std():.3f}")

# Entrenar el modelo final sobre TODO el dataset
print("\nEntrenando modelo final sobre dataset completo...")
pipeline_smote.fit(X, y)
print("✓ Modelo final entrenado")

---
## Celda 5 — Evaluar y comparar con modelo original

In [ ]:
# Celda 5
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, precision_score
)
from sklearn.model_selection import cross_val_predict
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler as SS
from xgboost import XGBClassifier as XGB

# Predicciones CV para matriz de confusión del modelo nuevo
y_pred_smote = cross_val_predict(pipeline_smote, X, y, cv=cv)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparación: modelo original vs SMOTE', fontsize=14)

# ---- Modelo original (sin corrección) ----
model_old = ImbPipeline([
    ('scaler', SS()),
    ('model', XGB(random_state=42, eval_metric='logloss', use_label_encoder=False))
])
y_pred_old = cross_val_predict(model_old, X, y, cv=cv)

cm_old = confusion_matrix(y, y_pred_old)
disp_old = ConfusionMatrixDisplay(cm_old, display_labels=['Sano', 'Parkinson'])
disp_old.plot(ax=axes[0], colorbar=False, cmap='Reds')
axes[0].set_title('Original (sin balanceo)')
f1_old = f1_score(y, y_pred_old)
rec_old = recall_score(y, y_pred_old)
axes[0].set_xlabel(f'F1={f1_old:.3f}  Recall={rec_old:.3f}')

# ---- Modelo con SMOTE ----
cm_smote = confusion_matrix(y, y_pred_smote)
disp_smote = ConfusionMatrixDisplay(cm_smote, display_labels=['Sano', 'Parkinson'])
disp_smote.plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Con SMOTE')
f1_new = f1_score(y, y_pred_smote)
rec_new = recall_score(y, y_pred_smote)
axes[1].set_xlabel(f'F1={f1_new:.3f}  Recall={rec_new:.3f}')

plt.tight_layout()
plt.show()

# Reporte completo
print("\n=== Reporte modelo con SMOTE ===")
print(classification_report(y, y_pred_smote, target_names=['Sano', 'Parkinson']))

print(f"\n{'Métrica':<12} {'Original':>10} {'SMOTE':>10} {'Mejora':>10}")
print("-" * 45)
for nombre, fn in [('Recall', recall_score), ('F1', f1_score), ('Precision', precision_score)]:
    v_old = fn(y, y_pred_old)
    v_new = fn(y, y_pred_smote)
    signo = "+" if v_new > v_old else ""
    print(f"{nombre:<12} {v_old:>10.3f} {v_new:>10.3f} {signo}{v_new-v_old:>9.3f}")

---
## Celda 5b — Comparar distribución de probabilidades (modelo viejo vs nuevo)

In [ ]:
# Celda 5b — Histograma de probabilidades: modelo original vs SMOTE
from sklearn.model_selection import cross_val_predict

# Obtener probabilidades con CV para ambos modelos
y_proba_old = cross_val_predict(model_old, X, y, cv=cv, method='predict_proba')[:, 1]
y_proba_new = cross_val_predict(pipeline_smote, X, y, cv=cv, method='predict_proba')[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de probabilidades predichas', fontsize=14)

# Modelo original
axes[0].hist(y_proba_old[y == 0], bins=20, alpha=0.7, label='Sano (real)', color='#2ecc71', edgecolor='white')
axes[0].hist(y_proba_old[y == 1], bins=20, alpha=0.7, label='Parkinson (real)', color='#e74c3c', edgecolor='white')
axes[0].axvline(0.5, color='gray', linestyle='--', linewidth=1.5, label='Umbral 0.5')
axes[0].set_xlabel('Probabilidad predicha de Parkinson')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Modelo Original (sin balanceo)')
axes[0].legend()

# Modelo con SMOTE
axes[1].hist(y_proba_new[y == 0], bins=20, alpha=0.7, label='Sano (real)', color='#2ecc71', edgecolor='white')
axes[1].hist(y_proba_new[y == 1], bins=20, alpha=0.7, label='Parkinson (real)', color='#e74c3c', edgecolor='white')
axes[1].axvline(0.5, color='gray', linestyle='--', linewidth=1.5, label='Umbral 0.5')
axes[1].axvline(0.65, color='blue', linestyle=':', linewidth=1.5, label='Umbral 0.65 (propuesto)')
axes[1].set_xlabel('Probabilidad predicha de Parkinson')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Modelo con SMOTE')
axes[1].legend()

plt.tight_layout()
plt.show()

# Análisis de umbrales
print("\n=== Análisis de umbrales (modelo con SMOTE) ===")
for threshold in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75]:
    y_pred_t = (y_proba_new >= threshold).astype(int)
    tn = np.sum((y == 0) & (y_pred_t == 0))
    fp = np.sum((y == 0) & (y_pred_t == 1))
    fn = np.sum((y == 1) & (y_pred_t == 0))
    tp = np.sum((y == 1) & (y_pred_t == 1))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"  Umbral {threshold:.2f}: Sensibilidad={sensitivity:.3f}, Especificidad={specificity:.3f}, FalsosPositivos={fp}")

---
## Celda 6 — Exportar los archivos `.sav` para FastAPI

*(Versión local: guarda directamente en `saved_models/`)*

In [ ]:
# Celda 6 — Versión local (sin Google Colab)
import pickle
import os

# El pipeline ya incluye scaler + smote + modelo
# Para producción solo necesitas el pipeline (sin SMOTE en inferencia)
# Extraemos scaler y model por separado para compatibilidad con MedDiag

scaler_final   = pipeline_smote.named_steps['scaler']
model_final    = pipeline_smote.named_steps['model']

# Asegurar que el directorio saved_models existe
os.makedirs('saved_models', exist_ok=True)

# Guardar en saved_models/
model_path  = 'saved_models/parkinsons_model_smote.sav'
scaler_path = 'saved_models/parkinsons_scaler_smote.sav'

pickle.dump(model_final,  open(model_path, 'wb'))
pickle.dump(scaler_final, open(scaler_path, 'wb'))

print("Archivos guardados:")
print(f"  {model_path}  — {os.path.getsize(model_path) / 1024:.1f} KB")
print(f"  {scaler_path} — {os.path.getsize(scaler_path) / 1024:.1f} KB")

# Verificar que funcionan
scaler_test = pickle.load(open(scaler_path, 'rb'))
model_test  = pickle.load(open(model_path, 'rb'))

# Test rápido con primera muestra
x_sample = X.iloc[[0]]
x_scaled  = scaler_test.transform(x_sample)
proba     = model_test.predict_proba(x_scaled)[0][1]
label     = 1 if proba >= 0.65 else 0   # umbral ajustado

print(f"\nTest con muestra 0:")
print(f"  Probabilidad PD : {proba:.3f}")
print(f"  Predicción (t=0.65): {'Parkinson' if label else 'Sano'}")
print(f"  Real: {'Parkinson' if y.iloc[0] == 1 else 'Sano'}")

print("\n✓ Modelos guardados localmente en saved_models/")
print("  Listos para usar con FastAPI/MedDiag")

---
## Cómo integrar los nuevos modelos en FastAPI

Los archivos `.sav` ya están en `saved_models/`. Actualiza tu servicio de inferencia:

```python
# En app/model_predict.py o tu servicio de Parkinson
import pickle
import numpy as np

# Cargar los nuevos archivos
scaler = pickle.load(open('saved_models/parkinsons_scaler_smote.sav', 'rb'))
model  = pickle.load(open('saved_models/parkinsons_model_smote.sav', 'rb'))

THRESHOLD = 0.65  # umbral ajustado para reducir falsos positivos

def predict_parkinson(features):
    X = np.array([features])
    X_scaled = scaler.transform(X)
    proba = model.predict_proba(X_scaled)[0][1]
    return {
        "prediction": int(proba >= THRESHOLD),
        "probability": round(float(proba), 4),
        "threshold": THRESHOLD,
        "model_version": "smote_v1"
    }
```

### Notas importantes:
- El umbral **0.65** se recomienda después de analizar el balance sensibilidad/especificidad
- SMOTE solo se usa durante entrenamiento, **no en inferencia**
- El scaler se extrajo del pipeline y se guarda aparte para compatibilidad con la arquitectura actual de MedDiag